# Notebook with minimal dataset

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(),'..'))

In [ ]:
import CIBUSmod as cm
import CIBUSmod.utils.plot as plot

import time
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt

In [ ]:
from CIBUSmod.utils.misc import inv_dict
from CIBUSmod.optimisation.indexed_matrix import IndexedMatrix
from itertools import product

In [ ]:
USE_MINI_DATASET = False

# Create session
session = cm.Session(
    name = 'mini',
    data_path = 'data' if USE_MINI_DATASET else '../data',
)

session.add_scenario(
    name='none',
    years=[0]
)

In [ ]:
%%time
# Instatiate Regions
regions = cm.Regions(
    par = cm.ParameterRetriever('Regions')
)

# Instantiate DemandAndConversions
demand = cm.DemandAndConversions(
    par = cm.ParameterRetriever('DemandAndConversions')
)

# Instantiate CropProduction
crops = cm.CropProduction(
    par = cm.ParameterRetriever('CropProduction'),
    index = regions.data_attr.get('x0_crops').index
)

# Instantiate AnimalHerds
# Each AnimalHerd object is stored in an indexed pandas.Series
herds = cm.make_herds(regions, sub_systems = {})

# Instantiate feed management
feed_mgmt = cm.FeedMgmt(
    herds = herds,
    par = cm.ParameterRetriever('FeedMgmt')
)

# Instantiate geo distributor
optproblem = cm.FeedDistributor(
    regions = regions,
    demand = demand,
    crops = crops,
    herds = herds,
    feed_mgmt = feed_mgmt,
    par = cm.ParameterRetriever('GeoDistributor')
)

self = optproblem

In [ ]:
%%time

cm.ParameterRetriever.update_all_parameter_values()
cm.ParameterRetriever.update_relation_tables()
regions.calculate(verbose=True)
demand.calculate(verbose=True)
crops.calculate(verbose=True)
for h in herds:
    h.calculate(verbose=True)

optproblem.make(use_cons=[1, 2, 10], verbose=True)

# Flip flag to solve by default
if False:
    optproblem.solve(verbose=True, apply_solution=True)
    feed_mgmt.calculate(verbose=True)
    
    session.store(
        'none', '0',
        demand, regions, crops, herds, optproblem
    )

In [ ]:
cm.plot.bar(
    session.get_attr('c','area',{'crop':['land_use',None],'region':None}).iloc[0].unstack('crop'),
    group_levels='land_use'
)

plt.show()   

In [ ]:
cm.plot.bar(
    session.get_attr('a','heads',['region','species']).iloc[0].unstack('species')
)
plt.show()